# 无标签预训练、TabPFN 微调与 boost 续训

当前源码安装：`pip install -e ".[net,boost,tune]"`。TabPFN 高级训练需要 Python 3.10+、tabpfn>=9.2。本例使用少量轮数验证接口、权重更新和序列化，不作为模型效果基准。

所有预训练仅接收特征 X。分类目标为 FPD，回归目标为放款金额；独立测试集不参与任何训练。TabPFN 使用 v2 权重，可通过环境变量 `HSCREDIT_TABPFN_CLASSIFIER_PATH` / `HSCREDIT_TABPFN_REGRESSOR_PATH` 指定本地文件。

In [ ]:
from pathlib import Path
from importlib.metadata import version
import os
import sys
import hashlib
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from IPython.display import display
from hscredit import ResNet, FTTransformer, TabM, TabPFN, XGBoost, LightGBM, CatBoost, NGBoost
from hscredit.core.models.net._preprocessing import TabularPreprocessor
from tabpfn import TabPFNClassifier as NativeClassifier, TabPFNRegressor as NativeRegressor
from tabpfn.constants import ModelVersion

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples/hscredit_yyp.xlsx").exists())
data = pd.read_excel(root / "examples/hscredit_yyp.xlsx")
features = ["衡枢鉴真分老客版", "近六个月非银多头机构数", "青云24", "商品类别"]
training, test = train_test_split(data, test_size=0.2, stratify=data.FPD, random_state=42)
train, valid = train_test_split(training, test_size=0.2, stratify=training.FPD, random_state=42)
X_train, X_valid, X_test = train[features], valid[features], test[features]
output_dir = root / ".audit_tmp" / "training-extensions" / "artifacts"
output_dir.mkdir(parents=True, exist_ok=True)
display(pd.DataFrame({"数据集": ["训练", "验证", "独立测试"], "样本数": [len(train), len(valid), len(test)]}))
display(pd.DataFrame({"组件": ["Python", "torch", "tabpfn"], "版本": [sys.version.split()[0], version("torch"), version("tabpfn")]}))

def parameter_digest(module):
    digest = hashlib.sha256()
    for name, parameter in module.named_parameters():
        digest.update(name.encode())
        digest.update(parameter.detach().cpu().numpy().tobytes())
    return digest.hexdigest()

## 1. 三种网络只用 X 做掩码预训练

数值重建使用标准化均方误差，类别使用交叉熵；仅计算被遮盖的真实观测值。TabM 逐成员计算损失再平均。下方初始化同种网络仅用于验收参数确实发生变化，正常使用只需 `pretrain`。

In [ ]:
encoders = {}
for cls in [ResNet, FTTransformer, TabM]:
    options = dict(d_block=32, n_blocks=2, max_epochs=4, batch_size=128, early_stopping_rounds=2, random_state=42)
    if cls is TabM:
        options["k"] = 4
    initial = cls(**options)
    initial.preprocessor_ = TabularPreprocessor().fit(X_train)
    devices = list(range(torch.cuda.device_count())) if torch.cuda.is_available() else []
    with torch.random.fork_rng(devices=devices):
        torch.manual_seed(42)
        initial._initialize_network()
    before = parameter_digest(initial._model)
    encoder = cls(**options).pretrain(X_train, eval_set=[X_valid], pretraining_ratio=0.4, max_epochs=3)
    assert parameter_digest(encoder.get_native_model()) != before
    assert encoder.get_embeddings(X_test).shape == (len(test), 32)
    assert not encoder._is_fitted
    encoder.save(output_dir / f"{cls.__name__}_预训练.pkl")
    restored = cls.load(output_dir / f"{cls.__name__}_预训练.pkl")
    np.testing.assert_allclose(restored.transform(X_test), encoder.transform(X_test))
    encoders[cls.__name__] = encoder
    display(encoder.get_learning_curve().tail())
print("三种网络均已仅用 X 更新参数，并完成预训练制品恢复")

## 2. 从预训练快照接续分类或回归
默认 `fit` 使用本实例快照；也可通过 `from_pretrained` 传入同类模型。字段处理保持预训练状态，来源模型不会被修改。

In [ ]:
results = []
trained_models = {}
for cls in [ResNet, FTTransformer, TabM]:
    source = encoders[cls.__name__]
    source_digest = parameter_digest(source.get_native_model())
    for task, target in [("binary", "FPD"), ("regression", "放款金额")]:
        options = source.get_params(deep=False)
        options.update(objective=task)
        model = cls(**options).fit(X_train, train[target], eval_set={"验证": (X_valid, valid[target])}, from_pretrained=source)
        prediction = model.predict(X_test)
        embedding = model.get_embeddings(X_test)
        assert prediction.shape == (len(test),) and embedding.shape == (len(test), 32)
        metrics = model.evaluate(X_test, test[target], metrics=["auc", "ks"] if task=="binary" else ["rmse", "mae"])
        results.append({"模型": cls.__name__, "任务": "二分类" if task=="binary" else "回归", **metrics})
        trained_models[f"{cls.__name__}_{task}"] = model
    assert parameter_digest(source.get_native_model()) == source_digest
display(pd.DataFrame(results))

## 3. TabPFN 的 X-only 字段预测预训练与可选微调

预训练从 X 生成遮盖任务并更新官方权重。回归重建数值；分类重建等频量化位，类别也按编码位重建。字段任务间传递已更新权重。该过程需要的目标来自 X 自身，不读取 FPD 或放款金额作为业务标签。

以下对原始官方权重、X-only 预训练权重、上下文拟合后的权重及监督微调权重分别校验。`finetune=False` 保持网络权重，`finetune=True` 更新权重。

In [ ]:
pfn_results = []
for task, kind, native_class, target in [("binary", "CLASSIFIER", NativeClassifier, "FPD"), ("regression", "REGRESSOR", NativeRegressor, "放款金额")]:
    native_options = {"device": "auto", "n_estimators": 1, "random_state": 42}
    path = os.environ.get(f"HSCREDIT_TABPFN_{kind}_PATH")
    if path:
        native_options["model_path"] = path
    initial = native_class.create_default_for_version(ModelVersion.V2, **native_options)
    initial._initialize_model_variables()  # 验收加载初始权重，不使用 X 或 y
    before = parameter_digest(initial.models_[0])
    model = TabPFN(objective=task, model_path=initial.model_path, n_estimators=1, random_state=42,
                   finetune_params={"model_version": "v2", "max_epochs": 1, "batch_size": 128,
                                    "validation_fraction": 0, "early_stopping_rounds": None,
                                    "use_lr_scheduler": False, "use_activation_checkpointing": False})
    model.pretrain(X_train, pretraining_ratio=0.5, max_epochs=1)
    pretrained_digest = parameter_digest(model.get_native_model().models_[0])
    assert pretrained_digest != before
    assert model.get_embeddings(X_test.iloc[:5]).shape[0] == 5
    model.save(output_dir / f"TabPFN_{task}_预训练.pkl")
    restored = TabPFN.load(output_dir / f"TabPFN_{task}_预训练.pkl")
    np.testing.assert_allclose(restored.get_embeddings(X_test.iloc[:5]), model.get_embeddings(X_test.iloc[:5]))
    model.fit(X_train, train[target], finetune=False)
    assert parameter_digest(model.get_native_model().models_[0]) == pretrained_digest
    model.fit(X_train, train[target], finetune=True)
    assert parameter_digest(model.get_native_model().models_[0]) != pretrained_digest
    metrics = model.evaluate(X_test, test[target], metrics=["auc", "ks"] if task=="binary" else ["rmse", "mae"])
    pfn_results.append({"模型": "TabPFN", "任务": "二分类" if task=="binary" else "回归", **metrics})
    trained_models[f"TabPFN_{task}"] = model
display(pd.DataFrame(pfn_results))

## 4. boost 追加训练

`continue_training=True` 使用当前已拟合模型；`init_model` 可指定另一个同类模型。`n_estimators/iterations` 表示本次新增轮数，普通 fit 仍从头训练。NGBoost 固定旧初始分布与学习率，避免改变已有树的贡献。

In [ ]:
def tree_count(model):
    native = model.get_native_model()
    if isinstance(model, XGBoost): return native.get_booster().num_boosted_rounds()
    if isinstance(model, LightGBM): return native.booster_.current_iteration()
    if isinstance(model, CatBoost): return native.tree_count_
    return len(native.base_models)

numeric_features = features[:3]
old, new = train.iloc[:len(train)//2], train.iloc[len(train)//2:]
boost_results = []
for cls, options in [(XGBoost, {"n_estimators": 5, "max_depth": 3}),
                     (LightGBM, {"n_estimators": 5, "max_depth": 3, "num_leaves": 8, "verbosity": -1}),
                     (CatBoost, {"iterations": 5, "depth": 3, "allow_writing_files": False}),
                     (NGBoost, {"n_estimators": 5, "natural_gradient": False})]:
    model = cls(n_jobs=1, random_state=42, validation_fraction=0, **options).fit(old[numeric_features], old.FPD)
    first_count = tree_count(model)
    model.fit(new[numeric_features[::-1]], new.FPD, continue_training=True)
    assert tree_count(model) == first_count + 5
    metrics = model.evaluate(test[numeric_features], test.FPD, metrics=["auc", "ks"])
    boost_results.append({"模型": cls.__name__, "原有轮数": first_count, "续训后轮数": tree_count(model), **metrics})
    trained_models[cls.__name__] = model
display(pd.DataFrame(boost_results))

## 5. 所有最终模型保存与恢复

In [ ]:
for name, model in trained_models.items():
    path = output_dir / f"{name}.pkl"
    model.save(path)
    restored = type(model).load(path)
    query = test[numeric_features] if name in {"XGBoost", "LightGBM", "CatBoost", "NGBoost"} else X_test
    np.testing.assert_allclose(restored.predict(query.iloc[:5]), model.predict(query.iloc[:5]))
    if hasattr(model, "get_embeddings"):
        np.testing.assert_allclose(restored.get_embeddings(query.iloc[:5]), model.get_embeddings(query.iloc[:5]))
print("X-only 权重更新、微调开关、编码器迁移、boost 追加轮数和保存恢复验证通过")